# U-Net — Segmentation route (Autonomous Twin)

Entraine un U-Net leger (3 niveaux) a segmenter la route (pixel blanc) dans la vue camera.

Dataset : 30 circuits generes proceduralement (seed 1000), split par circuit (25 train / 5 val = regle D10 du prof).

Cible : IoU > 0.90 sur le set val (circuits jamais vus en training).

Machine : Colab GPU (T4 gratuit) ou local avec CUDA.

## Pipeline rapide

1. Clone le repo Autonomous Twin (si on est sur Colab)
2. Install deps + genere dataset (headless, ~3 min)
3. Charge dataset PyTorch
4. Train U-Net avec BCE+Dice loss
5. Evaluate IoU par circuit
6. Export `models/unet_road.pth`

## 1. Setup environnement

In [ ]:
import os
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print(f'Colab : {IS_COLAB}')

if IS_COLAB:
    # Clone le repo (remplacer par ton fork si besoin)
    if not Path('autonomous-twin').exists():
        !git clone --branch dev/ml-nohlan-lorenzo https://github.com/CharlesCar0105/autonomous-twin.git
    os.chdir('autonomous-twin')
    !pip -q install pygame pyzmq opencv-python pillow
    # torch est deja installe sur Colab

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
print(f'ROOT : {ROOT}')

In [ ]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'mem    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go')

## 2. Generation du dataset

Si le dataset est deja present (ex. run local), skip ces cellules.

In [ ]:
data_dir = ROOT / 'data'
train_dir = data_dir / 'train'
val_dir = data_dir / 'val'

if not train_dir.exists() or not any(train_dir.iterdir()):
    print('[setup] generation circuits (seed 1000, 30 circuits)...')
    !python scripts/gen_circuits.py --n 30 --seed 1000
    print('[setup] collecte dataset (30s par circuit, stride 3)...')
    !python scripts/record_dataset.py --seconds 30 --frame-stride 3
else:
    print(f'[setup] dataset deja present : {train_dir}')

# Stats
train_frames = sum(1 for _ in train_dir.rglob('*.npz'))
val_frames = sum(1 for _ in val_dir.rglob('*.npz'))
print(f'train : {train_frames} frames')
print(f'val   : {val_frames} frames')

## 3. Dataset PyTorch

Charge les .npz, retourne (camera, mask) normalises. Augmentations pour la generalisation : flip horizontal, rotation legere, jitter luminosite/contraste.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image
import random


class RoadSegDataset(Dataset):
    """Paires (camera RGB, mask binaire) depuis les .npz de data/{split}/."""

    def __init__(self, root: Path, augment: bool = False):
        self.files = sorted(root.rglob('*.npz'))
        self.augment = augment
        if not self.files:
            raise RuntimeError(f'aucun .npz dans {root}')

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        d = np.load(self.files[idx])
        img = d['camera'].astype(np.float32) / 255.0      # (H, W, 3) [0,1]
        mask = d['mask'].astype(np.float32)                # (H, W) {0,1}

        img_t = torch.from_numpy(img).permute(2, 0, 1)     # (3, H, W)
        mask_t = torch.from_numpy(mask).unsqueeze(0)       # (1, H, W)

        if self.augment:
            # Flip horizontal (la voiture peut aller dans les 2 sens)
            if random.random() < 0.5:
                img_t = TF.hflip(img_t)
                mask_t = TF.hflip(mask_t)
            # Rotation legere +/-15 deg
            if random.random() < 0.5:
                ang = random.uniform(-15, 15)
                img_t = TF.rotate(img_t, ang, fill=[1.0, 1.0, 1.0])
                mask_t = TF.rotate(mask_t, ang, fill=[1.0])
            # Jitter brightness/contrast (la piste n'est pas toujours blanc pur)
            if random.random() < 0.5:
                img_t = TF.adjust_brightness(img_t, random.uniform(0.8, 1.2))
            if random.random() < 0.5:
                img_t = TF.adjust_contrast(img_t, random.uniform(0.8, 1.2))
            img_t = img_t.clamp(0, 1)

        return img_t, mask_t


train_ds = RoadSegDataset(train_dir, augment=True)
val_ds = RoadSegDataset(val_dir, augment=False)
print(f'train : {len(train_ds)}  val : {len(val_ds)}')

BATCH = 32
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# Visualise 6 paires (camera / mask)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for i in range(6):
    img, mask = train_ds[i * len(train_ds) // 6]
    axes[0, i].imshow(img.permute(1, 2, 0))
    axes[0, i].set_title(f'camera #{i}')
    axes[0, i].axis('off')
    axes[1, i].imshow(mask.squeeze(0), cmap='gray')
    axes[1, i].set_title('mask GT')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## 4. Architecture U-Net leger (3 niveaux)

~0.5M parametres, assez pour un domaine binaire 128x128. Blocks conv-BN-ReLU doubles, skip connections entre encodeur et decodeur.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


def conv_block(cin: int, cout: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, base: int = 32):
        super().__init__()
        self.enc1 = conv_block(3, base)
        self.enc2 = conv_block(base, base * 2)
        self.enc3 = conv_block(base * 2, base * 4)
        self.bottleneck = conv_block(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = conv_block(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = conv_block(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = conv_block(base * 2, base)

        self.out = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)  # logits


model = UNet(base=32).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'parametres : {n_params:,}')

## 5. Loss (BCE + Dice) et metrique IoU

In [ ]:
def dice_loss(logits: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    num = 2.0 * (probs * target).sum(dim=(1, 2, 3))
    den = probs.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + eps
    return (1.0 - num / den).mean()


def combo_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return 0.5 * F.binary_cross_entropy_with_logits(logits, target) + 0.5 * dice_loss(logits, target)


def iou(logits: torch.Tensor, target: torch.Tensor, thr: float = 0.5) -> float:
    preds = (torch.sigmoid(logits) > thr).float()
    inter = (preds * target).sum(dim=(1, 2, 3))
    union = ((preds + target).clamp(0, 1)).sum(dim=(1, 2, 3))
    return ((inter + 1e-6) / (union + 1e-6)).mean().item()

## 6. Training loop

In [ ]:
EPOCHS = 30
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

best_val_iou = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_iou': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for img, mask in train_loader:
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        logits = model(img)
        loss = combo_loss(logits, mask)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * img.size(0)
    tr_loss /= len(train_ds)

    model.eval()
    vl_loss = 0.0
    vl_iou = 0.0
    with torch.no_grad():
        for img, mask in val_loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)
            logits = model(img)
            vl_loss += combo_loss(logits, mask).item() * img.size(0)
            vl_iou += iou(logits, mask) * img.size(0)
    vl_loss /= len(val_ds)
    vl_iou /= len(val_ds)

    scheduler.step(vl_iou)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_iou'].append(vl_iou)

    marker = ''
    if vl_iou > best_val_iou:
        best_val_iou = vl_iou
        torch.save(model.state_dict(), ROOT / 'models' / 'unet_road.pth')
        marker = ' *'
    print(f'epoch {epoch:2d}  tr_loss={tr_loss:.4f}  vl_loss={vl_loss:.4f}  vl_iou={vl_iou:.4f}{marker}')

print(f'\nbest val IoU : {best_val_iou:.4f}  (cible D3 > 0.90)')

In [ ]:
# Courbes de training
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['val_iou'], label='val IoU')
axes[1].axhline(0.9, color='r', linestyle='--', alpha=0.5, label='cible 0.90')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('IoU'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Evaluation par circuit (verifie la generalisation D10)

IoU moyenne par circuit val : si tous >= 0.90 -> D10 valide.

In [ ]:
model.load_state_dict(torch.load(ROOT / 'models' / 'unet_road.pth'))
model.eval()

per_circuit = {}
with torch.no_grad():
    for npz_path in val_ds.files:
        circuit = npz_path.parent.name
        d = np.load(npz_path)
        img = torch.from_numpy(d['camera'].astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        mask = torch.from_numpy(d['mask'].astype(np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
        logits = model(img)
        score = iou(logits, mask)
        per_circuit.setdefault(circuit, []).append(score)

print(f"{'circuit':<12}  {'n_frames':>8}  {'IoU mean':>8}  {'IoU min':>8}")
print('-' * 44)
for c, scores in sorted(per_circuit.items()):
    print(f'{c:<12}  {len(scores):>8}  {np.mean(scores):>8.4f}  {np.min(scores):>8.4f}')

## 8. Visualisation predictions sur val

In [ ]:
import random as _rand

samples = _rand.sample(list(val_ds.files), 6)
fig, axes = plt.subplots(3, 6, figsize=(16, 9))
for i, p in enumerate(samples):
    d = np.load(p)
    img = d['camera'].astype(np.float32) / 255.0
    mask = d['mask']
    with torch.no_grad():
        t = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        pred = (torch.sigmoid(model(t)) > 0.5).cpu().squeeze().numpy()
    axes[0, i].imshow(img); axes[0, i].set_title(p.parent.name); axes[0, i].axis('off')
    axes[1, i].imshow(mask, cmap='gray'); axes[1, i].set_title('GT'); axes[1, i].axis('off')
    axes[2, i].imshow(pred, cmap='gray'); axes[2, i].set_title('pred'); axes[2, i].axis('off')
plt.tight_layout(); plt.show()

## 9. Export

Le meilleur modele est deja sauve dans `models/unet_road.pth` (automatiquement a chaque nouveau best IoU).

Si on est sur Colab, telecharger manuellement le .pth puis le mettre dans `models/` du repo local (gitignore par defaut, a commiter si on veut versionner).

In [ ]:
path = ROOT / 'models' / 'unet_road.pth'
print(f'modele sauve : {path}  ({path.stat().st_size / 1e6:.1f} Mo)')
if IS_COLAB:
    from google.colab import files
    files.download(str(path))